# Redrob AI — Intelligent Candidate Ranking System
### India Runs Hackathon 2026 | Track 1

**Author:** Praveena Ravichandran  
**Model:** BAAI/bge-base-en-v1.5  
**Approach:** Hybrid scoring — Semantic (50%) + Career (30%) + Behavioral (20%)  
**Key Features:** Honeypot detection, weighted profiles, exponential recency decay

## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

Mounted at /content/drive
Drive mounted!


## Cell 2 — Install Libraries

In [2]:
!pip install -q sentence-transformers scikit-learn python-docx
print('All libraries installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.7 MB/s eta 0:00:00
All libraries installed!


## Cell 3 — Import Everything

In [3]:
import json
import math
import pandas as pd
import numpy as np
from datetime import datetime, date
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('All imports successful!')

All imports successful!


## Cell 4 — Load Dataset
> Change `MODE = 'sample'` to `MODE = 'full'` for final submission

In [4]:

BASE_PATH = '/content/drive/MyDrive/redrob_hackathon/'

if MODE == 'full':
    with open(BASE_PATH + 'sample_candidates.json', 'r') as f:
        candidates = json.load(f)
    print(f'SAMPLE MODE: Loaded {len(candidates)} candidates')
else:
    candidates = []
    with open(BASE_PATH + 'candidates.jsonl', 'r') as f:
        for line in f:
            candidates.append(json.loads(line.strip()))
    print(f'FULL MODE: Loaded {len(candidates)} candidates')

print(f'First candidate: {candidates[0]["candidate_id"]}')
print(f'Title: {candidates[0]["profile"]["current_title"]}')
print(f'Keys available: {list(candidates[0].keys())}')

FULL MODE: Loaded 100000 candidates
First candidate: CAND_0000001
Title: Backend Engineer
Keys available: ['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals']


## Cell 5 — Load Job Description

In [5]:
doc = Document(BASE_PATH + 'job_description.docx')
job_text = ' '.join(p.text for p in doc.paragraphs if p.text.strip())

# Add JD keywords to boost matching accuracy
# These are core terms from the actual job description
jd_boost = [
    'senior AI engineer founding team',
    'python embeddings vector search ranking',
    'sentence transformers semantic search LLM',
    'evaluation framework retrieval augmented generation',
    'product company startup AI ML engineer',
    'Pune Noida India relocation'
] * 2

job_text_weighted = job_text + ' ' + ' '.join(jd_boost)

print(f'Job description loaded: {len(job_text)} characters')
print(f'Preview: {job_text[:200]}')

Job description loaded: 9564 characters
Preview: Job Description: Senior AI Engineer — Founding Team Company: Redrob AI (Series A AI-native talent intelligence platform) Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation ca


## Cell 6 — Honeypot Detection
> Removes fake/trap candidates that would hurt your score

In [6]:
def is_honeypot(candidate):
    """
    Detects fake/trap candidates in the dataset.
    Ranking honeypots high = score penalty.
    """
    p = candidate['profile']
    career = candidate.get('career_history', [])

    # Check 1: Impossible tenure (worked longer than company existed)
    for job in career:
        founded = job.get('company_founded_year', None)
        duration = job.get('duration_years', 0)
        if founded and duration:
            company_age = 2026 - int(founded)
            if int(duration) > company_age + 1:
                return True

    # Check 2: Future start dates
    for job in career:
        start = job.get('start_year', 0)
        if start and int(start) > 2026:
            return True

    # Check 3: Impossible experience
    exp = p.get('years_of_experience', 0)
    if exp > 45 or exp < 0:
        return True

    # Check 4: Everyone expert at everything (too good to be true)
    skills = candidate.get('skills', [])
    if len(skills) > 5:
        expert_count = sum(
            1 for s in skills
            if s.get('proficiency', '').lower() == 'expert'
        )
        if expert_count == len(skills):
            return True

    # Check 5: Negative or null candidate ID
    if not candidate.get('candidate_id'):
        return True

    return False


# Apply honeypot filter
clean_candidates = [c for c in candidates if not is_honeypot(c)]
removed = len(candidates) - len(clean_candidates)

print(f'Total candidates:  {len(candidates)}')
print(f'Honeypots removed: {removed}')
print(f'Clean candidates:  {len(clean_candidates)}')

Total candidates:  100000
Honeypots removed: 0
Clean candidates:  100000


## Cell 7 — Build Weighted Candidate Text Profiles
> Key fix: expert skills + recent jobs weighted higher. Stops wrong candidates (HR Manager, Graphic Designer) from ranking high.

In [7]:
# Services/outsourcing firms — JD specifically prefers product company experience
SERVICES_FIRMS = [
    'tcs', 'tata consultancy', 'infosys', 'wipro', 'accenture',
    'cognizant', 'capgemini', 'hcl', 'tech mahindra', 'mphasis',
    'hexaware', 'l&t infotech', 'ltimindtree', 'birlasoft'
]


def build_candidate_text(candidate):
    """
    Builds a weighted text profile for semantic matching.
    Important fields repeated to increase their embedding weight.
    """
    parts = []
    p = candidate['profile']

    # Current title — most important, repeat 3x
    title = p.get('current_title', '')
    parts.extend([title] * 3)

    # Headline — repeat 2x
    headline = p.get('headline', '')
    parts.extend([headline] * 2)

    # Professional summary
    parts.append(p.get('summary', ''))

    # Skills — expert/advanced repeated for higher weight
    for skill in candidate.get('skills', []):
        name = skill.get('name', '')
        level = skill.get('proficiency', '').lower()
        if level in ['expert', 'advanced']:
            parts.extend([name] * 2)
        else:
            parts.append(name)

    # Career history — most recent job weighted 3x
    for i, job in enumerate(candidate.get('career_history', [])):
        company = job.get('company', '').lower()
        is_services = any(f in company for f in SERVICES_FIRMS)
        # Recent + product company = most weight
        weight = 3 if i == 0 else 1
        if not is_services:
            weight += 1
        desc = (
            f"{job.get('title', '')} at "
            f"{job.get('company', '')} "
            f"{job.get('description', '')}"
        )
        parts.extend([desc] * weight)

    # Certifications
    for cert in candidate.get('certifications', []):
        parts.append(
            f"certified {cert.get('name', '')} "
            f"by {cert.get('issuer', '')}"
        )

    # Education
    for edu in candidate.get('education', []):
        parts.append(
            f"{edu.get('degree', '')} in "
            f"{edu.get('field', '')} from "
            f"{edu.get('institution', '')}"
        )

    return ' '.join(filter(None, parts))


print('Building weighted text profiles...')
candidate_texts = [build_candidate_text(c) for c in clean_candidates]
print(f'Built {len(candidate_texts)} profiles')
print(f'Sample profile (300 chars): {candidate_texts[0][:300]}')

Building weighted text profiles...
Built 100000 profiles
Sample profile (300 chars): Backend Engineer Backend Engineer Backend Engineer Backend Engineer | SQL, Spark, Cloud Backend Engineer | SQL, Spark, Cloud Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, S


## Cell 8 — AI Semantic Scoring
> UPGRADE: Using BAAI/bge-base-en-v1.5 instead of MiniLM — significantly more accurate for job matching

In [8]:
print('Loading BAAI/bge-base-en-v1.5 model...')
print('(Better than MiniLM — first load takes 2-3 mins)')

model = SentenceTransformer('BAAI/bge-base-en-v1.5')
print('Model loaded!')

print('\nEncoding job description...')
job_embedding = model.encode(
    [job_text_weighted],
    normalize_embeddings=True
)

print(f'Encoding {len(candidate_texts)} candidate profiles...')
print('(Progress bar below)')
candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

semantic_scores = cosine_similarity(job_embedding, candidate_embeddings)[0]

print(f'\nDone!')
print(f'Score range: {semantic_scores.min():.3f} to {semantic_scores.max():.3f}')
print(f'Average:     {semantic_scores.mean():.3f}')

Loading BAAI/bge-base-en-v1.5 model...
(Better than MiniLM — first load takes 2-3 mins)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!

Encoding job description...
Encoding 100000 candidate profiles...
(Progress bar below)


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]


Done!
Score range: 0.521 to 0.802
Average:     0.646


## Cell 9 — Behavioral Scoring
> UPGRADE: Exponential recency decay — candidates inactive 6+ months heavily penalized

In [9]:
def behavioral_score(candidate):
    """
    Scores candidate based on 5 Redrob behavioral signals.
    Uses exponential decay for recency — recent activity valued much more.
    """
    sig = candidate.get('redrob_signals', {})
    score = 0.0

    # 1. Open to work flag (25% weight)
    if sig.get('open_to_work_flag', False):
        score += 0.25

    # 2. Recruiter response rate (25% weight)
    rr = float(sig.get('recruiter_response_rate', 0))
    score += rr * 0.25

    # 3. Recency — exponential decay (25% weight)
    # Half-life = 90 days. Active today = 0.25, active 6mo ago = ~0.06
    last_active = sig.get('last_active_date', '2020-01-01')
    try:
        days_inactive = (
            date.today() -
            datetime.strptime(last_active, '%Y-%m-%d').date()
        ).days
        recency = math.exp(-days_inactive / 90)
        score += recency * 0.25
    except Exception:
        pass  # skip if date format issue

    # 4. Interview completion rate (15% weight)
    ic = float(sig.get('interview_completion_rate', 0))
    score += ic * 0.15

    # 5. Notice period — shorter is better for startup (10% weight)
    notice = int(sig.get('notice_period_days', 90))
    notice_score = max(0.0, 1.0 - (notice / 90))
    score += notice_score * 0.10

    return round(min(1.0, max(0.0, score)), 4)


# Test on first 5
print('Behavioral score samples:')
for c in clean_candidates[:5]:
    bs = behavioral_score(c)
    sig = c.get('redrob_signals', {})
    print(
        f"  {c['candidate_id']}: {bs:.3f} | "
        f"open={sig.get('open_to_work_flag')} | "
        f"response={sig.get('recruiter_response_rate', 0):.2f} | "
        f"last_active={sig.get('last_active_date', 'N/A')}"
    )

Behavioral score samples:
  CAND_0000001: 0.642 | open=True | response=0.34 | last_active=2026-05-20
  CAND_0000002: 0.469 | open=True | response=0.29 | last_active=2025-11-12
  CAND_0000003: 0.330 | open=False | response=0.46 | last_active=2026-03-21
  CAND_0000004: 0.207 | open=False | response=0.26 | last_active=2026-03-25
  CAND_0000005: 0.533 | open=True | response=0.37 | last_active=2025-10-01


## Cell 10 — Career Scoring
> Penalizes services firms, rewards product company + India location + GitHub activity

In [10]:
def career_score(candidate):
    """
    Scores candidate based on career fit for the JD:
    - Experience range 5-9 years (JD requirement)
    - Product company vs services firm background
    - India location (Pune/Noida target)
    - GitHub activity (JD values open source)
    - Relocation willingness
    """
    p = candidate['profile']
    sig = candidate.get('redrob_signals', {})
    score = 0.5  # neutral baseline

    # Experience — JD wants 5-9 years
    exp = p.get('years_of_experience', 0)
    if 5 <= exp <= 9:
        score += 0.25  # perfect match
    elif 3 <= exp < 5:
        score += 0.10  # slightly junior
    elif 9 < exp <= 12:
        score += 0.15  # slightly senior, still OK
    elif exp > 12:
        score += 0.05  # too senior for founding team role
    # exp < 3: no bonus — too junior

    # Penalize services firm background
    for job in candidate.get('career_history', []):
        company = job.get('company', '').lower()
        if any(f in company for f in SERVICES_FIRMS):
            score -= 0.20
            break

    # India location
    if p.get('country', '') == 'India':
        score += 0.10

    # Willing to relocate
    if sig.get('willing_to_relocate', False):
        score += 0.10

    # GitHub activity — JD values open source contribution
    github = sig.get('github_activity_score', -1)
    if github >= 70:
        score += 0.15
    elif github >= 50:
        score += 0.10
    elif github >= 20:
        score += 0.05

    return round(min(1.0, max(0.0, score)), 4)


# Test on first 5
print('Career score samples:')
for c in clean_candidates[:5]:
    cs = career_score(c)
    p = c['profile']
    print(
        f"  {c['candidate_id']}: {cs:.3f} | "
        f"title={p.get('current_title', '?')} | "
        f"exp={p.get('years_of_experience', 0)}yrs | "
        f"country={p.get('country', '?')}"
    )

Career score samples:
  CAND_0000001: 0.750 | title=Backend Engineer | exp=6.9yrs | country=Canada
  CAND_0000002: 0.450 | title=Operations Manager | exp=12.5yrs | country=India
  CAND_0000003: 0.300 | title=Customer Support | exp=1.1yrs | country=USA
  CAND_0000004: 0.500 | title=Marketing Manager | exp=3.8yrs | country=Australia
  CAND_0000005: 0.650 | title=Accountant | exp=11.0yrs | country=India


## Cell 11 — Combine Scores + Rank + Save
> Final formula: 50% semantic + 30% career + 20% behavioral

In [12]:
JD_KEYWORDS = [
    'python', 'embeddings', 'vector', 'ranking', 'ml', 'ai',
    'nlp', 'fastapi', 'llm', 'transformer', 'retrieval',
    'semantic', 'search', 'pytorch', 'tensorflow', 'rag'
]


def build_reasoning(c, sem, beh, car, final):
    """Builds a detailed reasoning string for each ranked candidate."""
    p = c['profile']
    sig = c.get('redrob_signals', {})
    skills = [s['name'] for s in c.get('skills', [])[:6]]
    matched_kw = [
        k for k in JD_KEYWORDS
        if k in ' '.join(skills).lower()
    ]
    return (
        f"Final:{final:.3f} "
        f"[Semantic:{sem:.2f}|Career:{car:.2f}|Behavioral:{beh:.2f}] "
        f"Title:{p.get('current_title','?')} "
        f"Exp:{p.get('years_of_experience',0)}yrs "
        f"Country:{p.get('country','?')} "
        f"JD_skills:{','.join(matched_kw) if matched_kw else 'none'} "
        f"Response:{sig.get('recruiter_response_rate',0):.0%} "
        f"Notice:{sig.get('notice_period_days',90)}days "
        f"OpenToWork:{sig.get('open_to_work_flag',False)}"
    )


# Score all candidates
print('Computing final scores for all candidates...')
results = []

for i, candidate in enumerate(clean_candidates):
    sem = float(semantic_scores[i])
    beh = behavioral_score(candidate)
    car = career_score(candidate)

    # Weighted hybrid score
    final = round(
        (0.50 * sem) + (0.30 * car) + (0.20 * beh),
        4
    )

    results.append({
        'candidate_id': candidate['candidate_id'],
        'score': final,
        'reasoning': build_reasoning(candidate, sem, beh, car, final)
    })

# Sort and assign ranks
df = pd.DataFrame(results)
df = df.sort_values('score', ascending=False).reset_index(drop=True)
df['rank'] = df.index + 1

# Take exactly top 100 (submission requirement)
top100 = df.head(100)[['candidate_id', 'rank', 'score', 'reasoning']]

print(f'\nTop 10 candidates:')
print(top100[['candidate_id', 'rank', 'score']].head(10).to_string(index=False))
print(f'\nScore range in top 100: {top100["score"].min():.4f} to {top100["score"].max():.4f}')

# Save to Drive
output_path = '/content/drive/MyDrive/redrob_hackathon/submission.csv'
top100.to_csv(output_path, index=False)
print(f'\nSaved {len(top100)} candidates to:\n{output_path}')

Computing final scores for all candidates...

Top 10 candidates:
candidate_id  rank  score
CAND_0002025     1 0.8628
CAND_0006567     2 0.8487
CAND_0046525     3 0.8438
CAND_0011687     4 0.8406
CAND_0018499     5 0.8340
CAND_0081846     6 0.8316
CAND_0071974     7 0.8288
CAND_0005538     8 0.8288
CAND_0062247     9 0.8242
CAND_0082086    10 0.8240

Score range in top 100: 0.7888 to 0.8628

Saved 100 candidates to:
/content/drive/MyDrive/redrob_hackathon/submission.csv


## Cell 12 — Validate Submission
> Run this before submitting! Checks all rules: 100 rows, correct columns, no duplicates, scores non-increasing

In [13]:
import subprocess

validate_path = '/content/drive/MyDrive/redrob_hackathon/validate_submission.py'
submission_path = '/content/drive/MyDrive/redrob_hackathon/submission.csv'

# Run official validator
result = subprocess.run(
    ['python', validate_path, '--submission', submission_path],
    capture_output=True, text=True
)

print('=== OFFICIAL VALIDATION RESULT ===')
print(result.stdout)
if result.stderr:
    print('Errors:', result.stderr)

# Manual checks as backup
print('=== MANUAL CHECKS ===')
df_check = pd.read_csv(submission_path)
print(f'Row count:     {len(df_check)} (must be 100)')
print(f'Columns:       {list(df_check.columns)}')
print(f'Duplicates:    {df_check["candidate_id"].duplicated().sum()} (must be 0)')
print(f'Ranks 1-100:   {df_check["rank"].min()} to {df_check["rank"].max()}')
scores_ok = all(
    df_check['score'].iloc[i] >= df_check['score'].iloc[i+1]
    for i in range(len(df_check)-1)
)
print(f'Scores non-increasing: {scores_ok} (must be True)')

if len(df_check) == 100 and df_check['candidate_id'].duplicated().sum() == 0 and scores_ok:
    print('\n ALL CHECKS PASSED! Ready to submit!')
else:
    print('\n SOME CHECKS FAILED! Fix before submitting!')

=== OFFICIAL VALIDATION RESULT ===
Usage: python validate_submission.py <participant_id>.csv

=== MANUAL CHECKS ===
Row count:     100 (must be 100)
Columns:       ['candidate_id', 'rank', 'score', 'reasoning']
Duplicates:    0 (must be 0)
Ranks 1-100:   1 to 100
Scores non-increasing: True (must be True)

 ALL CHECKS PASSED! Ready to submit!


## Cell 13 — Score Distribution Analysis
> Optional: Understand your results before submitting

In [14]:
print('=== SUBMISSION ANALYSIS ===')
print(f'Total candidates processed: {len(clean_candidates)}')
print(f'Honeypots removed: {len(candidates) - len(clean_candidates)}')
print(f'Top 100 score range: {top100["score"].min():.4f} to {top100["score"].max():.4f}')
print()

# Title distribution in top 10
print('Top 10 candidate titles:')
for _, row in top100.head(10).iterrows():
    cand = next(c for c in clean_candidates
                if c['candidate_id'] == row['candidate_id'])
    title = cand['profile'].get('current_title', 'N/A')
    exp = cand['profile'].get('years_of_experience', 0)
    country = cand['profile'].get('country', 'N/A')
    print(
        f"  Rank {row['rank']:2d}: {title:35s} "
        f"{exp:.1f}yrs | {country} | score={row['score']:.4f}"
    )

print()
print('If you see HR Managers or Graphic Designers in top 10, re-run with full dataset!')

=== SUBMISSION ANALYSIS ===
Total candidates processed: 100000
Honeypots removed: 0
Top 100 score range: 0.7888 to 0.8628

Top 10 candidate titles:
  Rank  1: Senior AI Engineer                  5.9yrs | India | score=0.8628
  Rank  2: Senior AI Engineer                  7.9yrs | India | score=0.8487
  Rank  3: Senior Machine Learning Engineer    6.1yrs | India | score=0.8438
  Rank  4: Senior NLP Engineer                 7.8yrs | India | score=0.8406
  Rank  5: Senior Machine Learning Engineer    7.2yrs | India | score=0.8340
  Rank  6: Lead AI Engineer                    6.7yrs | India | score=0.8316
  Rank  7: Senior AI Engineer                  7.8yrs | India | score=0.8288
  Rank  8: Senior AI Engineer                  5.9yrs | India | score=0.8288
  Rank  9: AI Engineer                         7.3yrs | India | score=0.8242
  Rank 10: Senior Software Engineer (ML)       6.0yrs | India | score=0.8240

If you see HR Managers or Graphic Designers in top 10, re-run with full dataset!
